# Parte 3 — Ecuaciones elípticas
## 3.4 Función de Green y fórmula de Poisson
### 3.4.01 Función de Green, fórmula de representación y estimaciones

**Fuente principal:** parte derecha de la página 8 y página 9 de las notas
manuscritas del archivo `Ecuación De Onda(1).pdf`.

Este notebook continúa después de
`03.2.02_Harnack_gradiente_analiticidad.ipynb`.

Se trabajan únicamente los siguientes bloques:

1. no unicidad de la solución fundamental;
2. segunda identidad de Green y fórmula de representación;
3. parte regular y función de Green de un dominio;
4. representación de soluciones del problema de Poisson-Dirichlet;
5. positividad y estimaciones de la función de Green;
6. fórmulas explícitas en la bola y el semiespacio.

El problema general de existencia para Dirichlet y el método de Perron quedan
para notebooks posteriores.

## Convenciones editoriales

- **Transcripción literal de las notas:** conserva el orden y la notación legible.
- **Aclaración:** explicita una hipótesis o una convención de signo.
- **Corrección editorial:** resuelve una ambigüedad sin modificar silenciosamente
  el contenido original.
- **Complemento:** añade una demostración o una fórmula exigida por el temario.
- Toda la matemática usa exclusivamente `$...$` y `$$...$$`.

## Páginas originales de las notas

### Página 8

![Página 8 — inicio de función de Green](../fuentes/pagina_08_notas.png)

### Página 9

![Página 9 — definición y estimaciones de Green](../fuentes/pagina_09_notas.png)

# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. animación de un polo móvil de la función de Green del disco;
2. descomposición entre solución fundamental, parte regular y Green;
3. reconstrucción mediante el núcleo de Poisson;
4. método de imágenes en el semiespacio;
5. verificación numérica de $0<G<\Phi$ en la bola tridimensional.

CuPy/CUDA se usa automáticamente en mallas densas cuando está disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    """Convierte un arreglo CuPy a NumPy; deja intacto un arreglo NumPy."""
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output

## Simulación 3.4.A — Polo móvil en el disco unitario

Identificamos $\mathbb R^2$ con $\mathbb C$. Para $z,y\in B_1(0)$,

$$
G_{B_1}(z,y)
=
\frac{1}{2\pi}
\log
\frac{|1-\overline y z|}{|z-y|}.
$$

La animación desplaza el polo $y$ dentro del disco. Deben observarse:

- la singularidad logarítmica en $z=y$;
- la positividad en el interior;
- la condición $G_{B_1}=0$ sobre la circunferencia;
- el movimiento de la parte regular para cancelar el dato de frontera de
  la solución fundamental.

In [ ]:
# ============================================================
# GREEN DEL DISCO CON POLO MÓVIL
# ============================================================

N = 620 if GPU_AVAILABLE else 340
x = xp.linspace(-1.03, 1.03, N)
y = xp.linspace(-1.03, 1.03, N)
X, Y = xp.meshgrid(x, y, indexing="xy")
R = xp.sqrt(X**2 + Y**2)
disk_mask = R < 1.0

n_frames = 240 if GPU_AVAILABLE else 150
angles = np.linspace(0.0, 2.0 * math.pi, n_frames, endpoint=False)
pole_radius = 0.48

poles = np.column_stack(
    [
        pole_radius * np.cos(angles),
        0.70 * pole_radius * np.sin(angles),
    ]
)


def green_disk_2d(X, Y, pole_x, pole_y):
    """
    Green del disco unitario:
        G(z,y) = (1/(2pi)) log(|1-conj(y)z|/|z-y|).
    """
    distance = xp.sqrt(
        (X - pole_x) ** 2
        + (Y - pole_y) ** 2
    )
    regular_factor = xp.sqrt(
        (1.0 - pole_x * X - pole_y * Y) ** 2
        + (pole_y * X - pole_x * Y) ** 2
    )

    eps = 1e-7
    G = (
        1.0
        / (2.0 * math.pi)
        * xp.log(
            xp.maximum(regular_factor, eps)
            / xp.maximum(distance, eps)
        )
    )

    return xp.where(disk_mask, G, xp.nan)


frames = []

for pole_x, pole_y in poles:
    G = green_disk_2d(X, Y, pole_x, pole_y)
    # Recorte puramente visual de la singularidad.
    frames.append(
        np.clip(to_cpu(G), 0.0, 1.10).astype(np.float32)
    )

frames = np.asarray(frames)

print("Malla:", N, "x", N)
print("Fotogramas:", len(frames))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7.2))

image = ax.imshow(
    frames[0],
    origin="lower",
    extent=[-1.03, 1.03, -1.03, 1.03],
    interpolation="bilinear",
    vmin=0.0,
    vmax=1.10,
)
fig.colorbar(image, ax=ax, label=r"$G_{B_1}(x,y)$")

boundary = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.5,
)
ax.add_patch(boundary)

pole_marker, = ax.plot(
    [poles[0, 0]],
    [poles[0, 1]],
    marker="o",
    linestyle="",
)

ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Función de Green del disco con polo móvil")


def update_green_disk(frame):
    image.set_data(frames[frame])
    pole_marker.set_data(
        [poles[frame, 0]],
        [poles[frame, 1]],
    )
    return image, pole_marker


animation = FuncAnimation(
    fig,
    update_green_disk,
    frames=len(frames),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.4.A_green_disco_polo_movil",
    fps=60,
    dpi=170,
    bitrate=14000,
)

plt.close(fig)

## Simulación 3.4.B — Solución fundamental, parte regular y Green

Para un polo fijo $y$, escribimos

$$
G_{B_1}(x,y)
=
\Phi(x-y)-\Gamma(x,y),
$$

donde

$$
\Phi(x-y)
=
-\frac{1}{2\pi}\log|x-y|
$$

y $\Gamma(\cdot,y)$ es armónica en el disco y coincide con $\Phi(\cdot-y)$
sobre la frontera.

Se presentan las tres funciones en figuras separadas.

In [ ]:
pole_x, pole_y = 0.38, -0.22

distance = xp.sqrt(
    (X - pole_x) ** 2
    + (Y - pole_y) ** 2
)
regular_factor = xp.sqrt(
    (1.0 - pole_x * X - pole_y * Y) ** 2
    + (pole_y * X - pole_x * Y) ** 2
)

eps = 1e-7
Phi = -1.0 / (2.0 * math.pi) * xp.log(
    xp.maximum(distance, eps)
)
Gamma = -1.0 / (2.0 * math.pi) * xp.log(
    xp.maximum(regular_factor, eps)
)
G = Phi - Gamma

Phi_cpu = np.where(to_cpu(disk_mask), to_cpu(Phi), np.nan)
Gamma_cpu = np.where(to_cpu(disk_mask), to_cpu(Gamma), np.nan)
G_cpu = np.where(to_cpu(disk_mask), to_cpu(G), np.nan)

for data, title, filename in [
    (
        np.clip(Phi_cpu, -0.2, 1.1),
        r"Solución fundamental $\Phi(x-y)$",
        "03.4.B_solucion_fundamental.png",
    ),
    (
        Gamma_cpu,
        r"Parte regular $\Gamma(x,y)$",
        "03.4.B_parte_regular.png",
    ),
    (
        np.clip(G_cpu, 0.0, 1.1),
        r"Función de Green $G=\Phi-\Gamma$",
        "03.4.B_green_descomposicion.png",
    ),
]:
    fig, ax = plt.subplots(figsize=(8.0, 6.8))
    image = ax.imshow(
        data,
        origin="lower",
        extent=[-1.03, 1.03, -1.03, 1.03],
        interpolation="bilinear",
    )
    fig.colorbar(image, ax=ax)
    circle = plt.Circle(
        (0.0, 0.0),
        1.0,
        fill=False,
        linewidth=1.5,
    )
    ax.add_patch(circle)
    ax.plot([pole_x], [pole_y], marker="o")
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_title(title)
    fig.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=220)
    plt.show()
    plt.close(fig)
    print(path.resolve())

# Error algebraico en la identidad G = Phi - Gamma.
decomposition_error = np.nanmax(
    np.abs(G_cpu - (Phi_cpu - Gamma_cpu))
)
print("Error máximo de descomposición:", decomposition_error)

## Simulación 3.4.C — Reconstrucción mediante el núcleo de Poisson

En el disco unitario,

$$
P(y,e^{i\theta})
=
-\frac{\partial G}{\partial\nu_x}
=
\frac{1-|y|^2}
{2\pi|e^{i\theta}-y|^2}.
$$

Para

$$
g(\theta)
=
\cos(3\theta)+0.4\sin(2\theta),
$$

la extensión armónica exacta es

$$
u(r,\varphi)
=
r^3\cos(3\varphi)
+
0.4r^2\sin(2\varphi).
$$

Se compara esta fórmula con la integral de Poisson.

In [ ]:
# ============================================================
# RECONSTRUCCIÓN CON EL NÚCLEO DE POISSON
# ============================================================

n_boundary = 12000 if GPU_AVAILABLE else 5000
theta_boundary = xp.linspace(
    0.0,
    2.0 * math.pi,
    n_boundary,
    endpoint=False,
)
boundary_x = xp.cos(theta_boundary)
boundary_y = xp.sin(theta_boundary)

g_boundary = (
    xp.cos(3.0 * theta_boundary)
    + 0.4 * xp.sin(2.0 * theta_boundary)
)

grid_n = 120 if GPU_AVAILABLE else 70
sample_axis = np.linspace(-0.86, 0.86, grid_n)
SX, SY = np.meshgrid(sample_axis, sample_axis, indexing="xy")
inside = SX**2 + SY**2 < 0.86**2

sample_points = np.column_stack([SX[inside], SY[inside]])
numerical_values = np.empty(len(sample_points), dtype=float)
exact_values = np.empty(len(sample_points), dtype=float)

for index, (px, py) in enumerate(sample_points):
    radius_sq = px**2 + py**2
    denominator = (
        (boundary_x - px) ** 2
        + (boundary_y - py) ** 2
    )
    poisson = (
        (1.0 - radius_sq)
        / (2.0 * math.pi * denominator)
    )

    # Integral sobre theta.
    numerical_values[index] = float(
        to_cpu(
            xp.mean(poisson * g_boundary)
            * 2.0
            * math.pi
        )
    )

    radius = math.sqrt(radius_sq)
    angle = math.atan2(py, px)

    exact_values[index] = (
        radius**3 * math.cos(3.0 * angle)
        + 0.4 * radius**2 * math.sin(2.0 * angle)
    )

errors = np.abs(numerical_values - exact_values)

print("Error máximo:", np.max(errors))
print("Error medio:", np.mean(errors))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    exact_values,
    numerical_values,
    s=8,
    alpha=0.5,
)
lower = min(exact_values.min(), numerical_values.min())
upper = max(exact_values.max(), numerical_values.max())
ax.plot([lower, upper], [lower, upper], linestyle="--")
ax.set_xlabel("solución exacta")
ax.set_ylabel("integral de Poisson")
ax.set_title("Reconstrucción del dato de Dirichlet")
ax.grid(True, alpha=0.3)
fig.tight_layout()

path = FIG_DIR / "03.4.C_reconstruccion_poisson.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

error_map = np.full_like(SX, np.nan, dtype=float)
error_map[inside] = errors

fig, ax = plt.subplots(figsize=(8, 6.8))
image = ax.imshow(
    error_map,
    origin="lower",
    extent=[
        sample_axis.min(),
        sample_axis.max(),
        sample_axis.min(),
        sample_axis.max(),
    ],
    interpolation="nearest",
)
fig.colorbar(image, ax=ax, label="error absoluto")
ax.set_aspect("equal")
ax.set_xlabel(r"$y_1$")
ax.set_ylabel(r"$y_2$")
ax.set_title("Error espacial de la fórmula de Poisson")
fig.tight_layout()

path = FIG_DIR / "03.4.C_error_poisson.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

## Simulación 3.4.D — Método de imágenes en el semiespacio

Sea

$$
\mathbb H
=
\{x=(x_1,x_2)\in\mathbb R^2:x_2>0\}.
$$

Si $y=(y_1,y_2)$ y $y^*=(y_1,-y_2)$ es su reflexión, entonces

$$
G_{\mathbb H}(x,y)
=
\Phi(x-y)-\Phi(x-y^*).
$$

Las dos singularidades tienen signos opuestos y producen valor cero sobre
$x_2=0$.

In [ ]:
# ============================================================
# GREEN DEL SEMIPLANO POR IMÁGENES
# ============================================================

nx = 650 if GPU_AVAILABLE else 360
ny = 420 if GPU_AVAILABLE else 250

hx = xp.linspace(-2.5, 2.5, nx)
hy = xp.linspace(0.0, 2.7, ny)
HX, HY = xp.meshgrid(hx, hy, indexing="xy")

source = (0.45, 1.05)
image_source = (source[0], -source[1])

r_source = xp.sqrt(
    (HX - source[0]) ** 2
    + (HY - source[1]) ** 2
)
r_image = xp.sqrt(
    (HX - image_source[0]) ** 2
    + (HY - image_source[1]) ** 2
)

G_half = (
    1.0
    / (2.0 * math.pi)
    * xp.log(
        xp.maximum(r_image, 1e-8)
        / xp.maximum(r_source, 1e-8)
    )
)

G_half_cpu = np.clip(to_cpu(G_half), 0.0, 1.1)

fig, ax = plt.subplots(figsize=(9, 6.5))
image = ax.imshow(
    G_half_cpu,
    origin="lower",
    extent=[-2.5, 2.5, 0.0, 2.7],
    interpolation="bilinear",
    aspect="auto",
)
fig.colorbar(image, ax=ax, label=r"$G_{\mathbb H}(x,y)$")
ax.axhline(0.0, linewidth=1.5)
ax.plot([source[0]], [source[1]], marker="o", label="polo")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Método de imágenes en el semiplano")
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.4.D_green_semiplano.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

boundary_error = np.max(np.abs(to_cpu(G_half[0, :])))
print("Máximo valor absoluto sobre la frontera:", boundary_error)
print(path.resolve())

## Simulación 3.4.E — Estimación $0<G<\Phi$ en la bola de $\mathbb R^3$

Para la bola unitaria y $n=3$,

$$
\Phi(z)=\frac{1}{4\pi|z|}.
$$

Si

$$
y^*=\frac{y}{|y|^2},
$$

entonces

$$
G_{B_1}(x,y)
=
\Phi(x-y)
-
|y|^{-1}\Phi(x-y^*).
$$

Se evalúa la función sobre el plano $x_3=0$ y se verifica numéricamente

$$
0<G_{B_1}(x,y)<\Phi(x-y).
$$

In [ ]:
# ============================================================
# ESTIMACIÓN EN LA BOLA TRIDIMENSIONAL
# ============================================================

n_slice = 620 if GPU_AVAILABLE else 340
axis = xp.linspace(-0.995, 0.995, n_slice)
BX, BY = xp.meshgrid(axis, axis, indexing="xy")
BZ = xp.zeros_like(BX)
ball_mask = BX**2 + BY**2 + BZ**2 < 1.0

pole = xp.asarray([0.32, -0.18, 0.26])
pole_norm_sq = float(to_cpu(xp.sum(pole**2)))
image_pole = pole / pole_norm_sq
pole_norm = math.sqrt(pole_norm_sq)

distance_pole = xp.sqrt(
    (BX - pole[0]) ** 2
    + (BY - pole[1]) ** 2
    + (BZ - pole[2]) ** 2
)
distance_image = xp.sqrt(
    (BX - image_pole[0]) ** 2
    + (BY - image_pole[1]) ** 2
    + (BZ - image_pole[2]) ** 2
)

Phi_3 = 1.0 / (
    4.0 * math.pi * xp.maximum(distance_pole, 1e-8)
)
correction_3 = (
    1.0
    / pole_norm
    / (
        4.0
        * math.pi
        * xp.maximum(distance_image, 1e-8)
    )
)
G_3 = Phi_3 - correction_3

valid_G = G_3[ball_mask]
valid_Phi = Phi_3[ball_mask]

print("mínimo de G:", float(to_cpu(xp.min(valid_G))))
print(
    "máximo de G-Phi:",
    float(to_cpu(xp.max(valid_G - valid_Phi))),
)

ratio = xp.where(
    ball_mask,
    G_3 / Phi_3,
    xp.nan,
)
ratio_cpu = to_cpu(ratio)

fig, ax = plt.subplots(figsize=(8, 6.8))
image = ax.imshow(
    ratio_cpu,
    origin="lower",
    extent=[-0.995, 0.995, -0.995, 0.995],
    interpolation="bilinear",
    vmin=0.0,
    vmax=1.0,
)
fig.colorbar(image, ax=ax, label=r"$G/\Phi$")
circle = plt.Circle(
    (0.0, 0.0),
    1.0,
    fill=False,
    linewidth=1.5,
)
ax.add_patch(circle)
ax.set_aspect("equal")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title(r"Verificación de $0<G<\Phi$ en una sección")
fig.tight_layout()

path = FIG_DIR / "03.4.E_estimacion_green_bola_3d.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

# 3.4.1 Solución fundamental y segunda identidad de Green

### **Transcripción literal de las notas**

**4.3 Función de Green y fórmula de Poisson.**

$$
\Phi(x-y)
=
\begin{cases}
-\dfrac{1}{2\pi}\log|x-y|,
& n=2,\\[2mm]
\dfrac{1}{(n-2)\omega_n|x-y|^{n-2}},
& n\geq3.
\end{cases}
$$

$\Phi$ no es única, en el sentido de que si $w$ es armónica, entonces

$$
\widetilde\Phi(x-y)
=
\Phi(x-y)+w(x)
$$

también tiene la misma singularidad fundamental.

En la página aparece además la identidad

$$
\int_\Omega
\left(
u\Delta v-v\Delta u
\right)dx
=
\int_{\partial\Omega}
\left(
u\frac{\partial v}{\partial\nu}
-
v\frac{\partial u}{\partial\nu}
\right)dS.
$$

## **Definición 3.4.1 (Solución fundamental de $-\Delta$).**

Una distribución $\Phi\in\mathcal D'(\mathbb R^n)$ es una solución fundamental
de $-\Delta$ si

$$
-\Delta\Phi=\delta_0.
$$

## **Proposición 3.4.2 (No unicidad de una solución fundamental).**

Sea $w\in C^2(\mathbb R^n)$ armónica. Si $\Phi$ es una solución fundamental de
$-\Delta$, entonces

$$
\widetilde\Phi=\Phi+w
$$

también satisface

$$
-\Delta\widetilde\Phi=\delta_0.
$$

### Demostración

Por linealidad,

$$
-\Delta(\Phi+w)
=
-\Delta\Phi-\Delta w
=
\delta_0.
$$

$\square$

## **Teorema 3.4.3 (Segunda identidad de Green).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado con frontera de clase $C^1$.
Sean

$$
u,v\in C^2(\Omega)\cap C^1(\overline\Omega).
$$

Entonces

$$
\int_\Omega
\left(
u\Delta v-v\Delta u
\right)dx
=
\int_{\partial\Omega}
\left(
u\frac{\partial v}{\partial\nu}
-
v\frac{\partial u}{\partial\nu}
\right)dS,
$$

donde $\nu$ es la normal unitaria exterior a $\partial\Omega$.

### Demostración

Aplicamos el teorema de la divergencia al campo

$$
F=u\nabla v-v\nabla u.
$$

Como

$$
\operatorname{div}F
=
u\Delta v-v\Delta u,
$$

se obtiene la identidad.

$\square$

## **Teorema 3.4.4 (Fórmula de representación con la solución fundamental).**

Sea $\Omega\subset\mathbb R^n$ un dominio acotado con frontera $C^1$ y sea

$$
u\in C^2(\Omega)\cap C^1(\overline\Omega).
$$

Para cada $y\in\Omega$,

$$
\boxed{
u(y)
=
-\int_\Omega
\Phi(x-y)\Delta u(x)\,dx
+
\int_{\partial\Omega}
\left[
\Phi(x-y)\frac{\partial u}{\partial\nu_x}(x)
-
u(x)\frac{\partial\Phi}{\partial\nu_x}(x-y)
\right]dS_x.
}
$$

### Demostración añadida

Se aplica la segunda identidad de Green en el dominio perforado

$$
\Omega_\varepsilon
=
\Omega\setminus\overline{B_\varepsilon(y)}
$$

a $u$ y $\Phi(\cdot-y)$. En $\Omega_\varepsilon$,

$$
\Delta_x\Phi(x-y)=0.
$$

Al hacer $\varepsilon\downarrow0$, el término sobre la esfera pequeña que contiene
la derivada normal de $\Phi$ converge a $u(y)$, mientras que el término con
$\Phi\,\partial_\nu u$ tiende a cero. El término de frontera exterior permanece
sin cambios y se obtiene la fórmula.

$\square$

### **Aclaración de signos.**

La convención usada en todas las notas es

$$
-\Delta\Phi=\delta_0.
$$

Cambiar a una convención para $\Delta\Phi=\delta_0$ modifica los signos de la
representación.

### Ejercicios — Sección 3.4.1

1. Derive la primera identidad de Green a partir del teorema de la divergencia.

2. Complete todos los límites sobre $\partial B_\varepsilon(y)$ en la demostración
   de la fórmula de representación.

3. Verifique la representación para

   $$
   u(x)=|x|^2
   $$

   en una bola.

4. **Tipo Examen General.** Use dos aplicaciones sucesivas de la segunda identidad
   de Green para obtener una fórmula de representación asociada al biharmónico
   $\Delta^2$.

# 3.4.2 Parte regular y función de Green

### **Transcripción literal de las notas**

**Definición.**

$$
G_\Omega(x,y)
=
\Phi(x-y)-\Gamma(x,y),
\qquad
x,y\in\Omega,
\quad
x\neq y,
$$

es la función de Green de $\Omega$.

## **Definición 3.4.5 (Parte regular).**

Sea $\Omega$ un dominio para el cual puede resolverse el problema de Dirichlet
con dato continuo. Para $y\in\Omega$, la **parte regular**
$\Gamma(\cdot,y)$ es la solución armónica de

$$
\begin{cases}
\Delta_x\Gamma(x,y)=0,
& x\in\Omega,\\
\Gamma(x,y)=\Phi(x-y),
& x\in\partial\Omega.
\end{cases}
$$

## **Definición 3.4.6 (Función de Green de un dominio).**

La función de Green de $\Omega$ para el operador $-\Delta$ es

$$
G_\Omega(x,y)
=
\Phi(x-y)-\Gamma(x,y).
$$

Para cada polo fijo $y\in\Omega$ satisface

$$
\begin{cases}
-\Delta_xG_\Omega(x,y)=\delta_y,
& x\in\Omega,\\
G_\Omega(x,y)=0,
& x\in\partial\Omega.
\end{cases}
$$

### **Aclaración de existencia.**

La definición presupone que existe la parte regular. Esta existencia está
vinculada con la solvencia del problema de Dirichlet y no se da por sentada en
dominios arbitrarios.

## **Teorema 3.4.7 (Simetría de la función de Green).**

Supongamos que $\Omega$ admite una función de Green. Entonces

$$
G_\Omega(x,y)=G_\Omega(y,x)
$$

para $x\neq y$.

### Demostración añadida

Se aplica la segunda identidad de Green a

$$
G_\Omega(\cdot,x)
\quad\text{y}\quad
G_\Omega(\cdot,y)
$$

en $\Omega$ con pequeñas bolas alrededor de $x$ y $y$ removidas.
Los términos de la frontera exterior se anulan porque ambas funciones son cero
sobre $\partial\Omega$. Los límites en las esferas pequeñas producen
$G_\Omega(x,y)$ y $G_\Omega(y,x)$.

$\square$

# 3.4.3 Fórmula de Green para Poisson-Dirichlet

## **Teorema 3.4.8 (Representación mediante la función de Green).**

Sea $\Omega$ un dominio acotado con frontera suficientemente regular y con función
de Green. Supongamos que

$$
u\in C^2(\Omega)\cap C^1(\overline\Omega)
$$

resuelve

$$
\begin{cases}
-\Delta u=f,
& \text{en }\Omega,\\
u=g,
& \text{sobre }\partial\Omega.
\end{cases}
$$

Entonces

$$
\boxed{
u(y)
=
\int_\Omega
G_\Omega(x,y)f(x)\,dx
-
\int_{\partial\Omega}
g(x)
\frac{\partial G_\Omega}{\partial\nu_x}(x,y)
\,dS_x.
}
$$

### Demostración

Se sustituye $\Phi(\cdot-y)$ por $G_\Omega(\cdot,y)$ en la fórmula de
representación. Como

$$
G_\Omega(x,y)=0
\qquad
\text{para }x\in\partial\Omega,
$$

desaparece el término que contiene $\partial_\nu u$. Además,

$$
-\Delta u=f.
$$

$\square$

## **Definición 3.4.9 (Núcleo de Poisson asociado a Green).**

Para $x\in\partial\Omega$ y $y\in\Omega$ se define

$$
P_\Omega(y,x)
=
-\frac{\partial G_\Omega}{\partial\nu_x}(x,y).
$$

Cuando $P_\Omega\geq0$, la representación de una función armónica se escribe

$$
u(y)
=
\int_{\partial\Omega}
P_\Omega(y,x)g(x)\,dS_x.
$$

### Ejercicios — Sección 3.4.3

1. Demuestre que, cuando $f=0$ y $g\equiv1$,

   $$
   \int_{\partial\Omega}
   P_\Omega(y,x)\,dS_x=1.
   $$

2. Use la representación para demostrar nuevamente unicidad del problema de
   Dirichlet.

3. Sea $f\geq0$ y $g\geq0$. Explique cómo la positividad de Green implica
   $u\geq0$.

4. **Tipo Examen General.** Explique a detalle qué es una función de Green y cómo
   permite construir una solución explícita. Proporcione una fórmula cerrada para
   un dominio concreto y verifique sus propiedades.

# 3.4.4 Positividad y estimaciones

### **Transcripción literal de las notas**

Sea $\Omega\subset\mathbb R^n$ abierto y acotado. Para $x\neq y$,
$x,y\in\Omega$:

si $n\geq3$,

$$
0<G_\Omega(x,y)<\Phi(x-y);
$$

si $n=2$,

$$
0<G_\Omega(x,y)
<
\Phi(x-y)
+
\frac{1}{2\pi}\log(\operatorname{diam}\Omega).
$$

Aquí

$$
\operatorname{diam}\Omega
=
\sup\{|x-y|:x,y\in\Omega\}.
$$

## **Teorema 3.4.10 (Positividad y cota superior de Green).**

Supongamos que $\Omega$ es un dominio acotado con función de Green.

1. Si $n\geq3$, entonces

   $$
   0<G_\Omega(x,y)\leq\Phi(x-y).
   $$

2. Si $n=2$, entonces

   $$
   0<G_\Omega(x,y)
   \leq
   \Phi(x-y)
   +
   \frac{1}{2\pi}\log(\operatorname{diam}\Omega).
   $$

Las desigualdades son estrictas para $x\neq y$ bajo las hipótesis usuales de
regularidad y conexidad.

### Demostración añadida

Fijemos $y\in\Omega$.

Para $n\geq3$, la parte regular $\Gamma(\cdot,y)$ es armónica y tiene dato de
frontera positivo:

$$
\Gamma(x,y)=\Phi(x-y)>0.
$$

El principio del mínimo implica $\Gamma>0$ en $\Omega$. Por tanto,

$$
G_\Omega=\Phi-\Gamma<\Phi.
$$

La positividad de $G_\Omega$ se obtiene aplicando el principio del mínimo en
$\Omega\setminus\overline{B_\varepsilon(y)}$ y haciendo
$\varepsilon\downarrow0$.

En dimensión dos, sea

$$
M=
\frac{1}{2\pi}
\log(\operatorname{diam}\Omega).
$$

Sobre la frontera,

$$
\Phi(x-y)+M\geq0.
$$

Como $\Gamma+M$ es armónica y no negativa en la frontera,

$$
\Gamma+M\geq0.
$$

Por tanto,

$$
G_\Omega
=
\Phi-\Gamma
\leq
\Phi+M.
$$

$\square$

### Ejercicios — Sección 3.4.4

1. Justifique con detalle la positividad de Green usando dominios perforados.

2. Determine cuándo la constante logarítmica de la cota bidimensional es positiva
   o negativa y explique por qué la desigualdad sigue siendo válida.

3. Pruebe que la parte regular es positiva en dimensión $n\geq3$.

4. **Tipo Examen General.** Sea $\Omega$ un dominio exterior. Analice qué cambios
   se requieren en las estimaciones de Green y qué condición en el infinito
   reemplaza al dato sobre una frontera exterior.

# 3.4.5 Fórmulas explícitas: bola y semiespacio

## **Proposición 3.4.11 (Función de Green de la bola unitaria).**

Sea

$$
B_1=\{x\in\mathbb R^n:|x|<1\}.
$$

Para $y\neq0$ definimos el punto inverso

$$
y^*=\frac{y}{|y|^2}.
$$

Si $n\geq3$,

$$
G_{B_1}(x,y)
=
\Phi(x-y)
-
|y|^{2-n}\Phi(x-y^*).
$$

En dimensión dos, identificando $\mathbb R^2$ con $\mathbb C$,

$$
G_{B_1}(z,y)
=
\frac{1}{2\pi}
\log
\frac{|1-\overline y z|}{|z-y|}.
$$

### Verificación

La segunda parte de cada fórmula es armónica en $x\in B_1$, porque el polo
reflejado está fuera de la bola. Además, sobre $|x|=1$ las dos contribuciones
coinciden, de modo que $G_{B_1}=0$.

## **Corolario 3.4.12 (Núcleo de Poisson de la bola).**

Para $x\in\partial B_1$ y $y\in B_1$,

$$
P_{B_1}(y,x)
=
\frac{1-|y|^2}
{\omega_n|x-y|^n}.
$$

## **Proposición 3.4.13 (Función de Green del semiespacio).**

Sea

$$
\mathbb H
=
\{x=(x',x_n)\in\mathbb R^n:x_n>0\}.
$$

Para $y=(y',y_n)\in\mathbb H$ definimos

$$
y^*=(y',-y_n).
$$

Entonces

$$
G_{\mathbb H}(x,y)
=
\Phi(x-y)-\Phi(x-y^*).
$$

La igualdad de distancias

$$
|x-y|=|x-y^*|
$$

para $x\in\partial\mathbb H$ garantiza que $G_{\mathbb H}=0$ en la frontera.

## **Corolario 3.4.14 (Núcleo de Poisson del semiespacio).**

Para $\xi\in\mathbb R^{n-1}$,

$$
P_{\mathbb H}(y,\xi)
=
\frac{2}{\omega_n}
\frac{y_n}
{\left(|y'-\xi|^2+y_n^2\right)^{n/2}}.
$$

### Ejercicios — Sección 3.4.5

1. Verifique directamente que la fórmula de Green de la bola se anula en la
   frontera.

2. Derive el núcleo de Poisson de la bola mediante la derivada normal de Green.

3. Compruebe que el núcleo de Poisson del semiespacio tiene integral uno.

4. **Tipo Examen General.** Obtenga la función de Green de una bola de radio $R$
   y centro $x_0$ a partir de la fórmula de la bola unitaria.

5. **Tipo Examen General.** Use el método de imágenes para encontrar la función
   de Green de un semiespacio y resolver un problema de Dirichlet con dato de
   frontera suficientemente regular.

# Control de cobertura y estado del capítulo

## Contenido cubierto

- no unicidad de la solución fundamental;
- segunda identidad de Green;
- fórmula de representación con $\Phi$;
- parte regular y función de Green;
- simetría;
- representación de Poisson-Dirichlet;
- núcleo de Poisson;
- positividad y estimaciones;
- fórmulas explícitas en la bola y el semiespacio;
- simulaciones del polo, método de imágenes y reconstrucción de frontera.

## Complementos añadidos

Las notas contienen los enunciados esenciales, pero no desarrollan:

- existencia de la parte regular;
- simetría de Green;
- deducción de la representación para Poisson;
- fórmulas explícitas de bola y semiespacio;
- demostraciones de las estimaciones.

Estos puntos se añadieron y quedaron marcados como complementos.

## Pendiente inmediato

El siguiente notebook de la cola es

$$
\texttt{03.5.01\_Problema\_de\_Dirichlet.ipynb}.
$$

Se trabajarán la formulación clásica del problema, puntos regulares, barreras y
existencia en dominios donde la solución pueda construirse explícitamente.